# cli

> The stream-protocol frontend: the service on stdin/stdout for token-reading clients

In [ ]:
#| default_exp cli

The `clikernel` command: the delimiter-framed stdin/stdout protocol v1 established (documented in the README, rationale unchanged — a client that reads stdout as tokens wants no echo, a cheap ack byte, and a per-process random delimiter to read until). The protocol machinery ports from v1 verbatim; underneath, the process is now a thin client of a gateway kernel. Run bare it creates a kernel and stops it again on exit — whoever ran the command made that decision by running it — while `--kernel` attaches to an existing kernel and leaves it exactly as found. Ctrl-C during a long cell translates into a kernel interrupt, jupyter-console style, instead of killing the process.


In [ ]:
#| export
import asyncio, secrets, signal, string, sys, termios, threading, traceback, tty
from fastcore.utils import *
from fastcore.script import call_parse
from clikernel.core import Client


In [ ]:
from fastcore.test import *
import httpx, os, shutil, tempfile
from clikernel.cli import _MARKER
from rustygate.tools import start_gateway


## The protocol

In [ ]:
#| export
_ALPHANUM = string.ascii_letters + string.digits
_MULTILINE = "--"
_MARKER = "loading complete. session delimiter:"


def _new_delim(): return "--" + ''.join(secrets.choice(_ALPHANUM) for _ in range(5))


def _read_block(stdin, delim):
    lines = []
    for line in stdin:
        if line.rstrip("\n") == delim: return "".join(lines), None
        lines.append(line)
    return "", f"missing block terminator: {delim}"


def fmt_error(tag, text):
    nl = '' if text.endswith('\n') else '\n'
    return f"<{tag}>\n{text}{nl}</{tag}>"


def _write_response(delim, body=None):
    if body: print(body, end='' if body.endswith('\n') else '\n', flush=True)
    print(delim, flush=True)


def _next_line(stdin):
    "Read one line; when not a TTY, SIGINT while idle means 'interrupt execution', not 'kill the worker', so ignore it"
    while True:
        try: return stdin.readline()
        except KeyboardInterrupt:
            if stdin.isatty(): raise


def _tty_clear(stream, idx, mask, cc=None):
    "Clear `mask` bits in termios field `idx` when `stream` is a TTY, with optional `cc` char overrides; returns state for `_restore_termios`"
    if not stream.isatty(): return None
    fd = stream.fileno()
    attrs = termios.tcgetattr(fd)
    new_attrs = attrs[:]
    new_attrs[idx] &= ~mask
    if cc:
        new_attrs[6] = attrs[6][:]
        for k, v in cc.items(): new_attrs[6][k] = v
    termios.tcsetattr(fd, termios.TCSADRAIN, new_attrs)
    return fd, attrs


def _restore_termios(state):
    if state: termios.tcsetattr(state[0], termios.TCSADRAIN, state[1])


def serve_stream(
    execute,          # Callable `code -> str`: run one request, returning the rendered response body
    info="",          # Server info announced between the loading lines (forwarded to mcp `instructions`)
    should_exit=None  # Callable checked after each request; truthy stops the worker
):
    "Run the stream protocol on stdin/stdout: announce `info` and the session delimiter, then ack each request with '.', respond with `execute(code)`, and end each response with the delimiter"
    # ONLCR off so protocol output stays bare LF; ECHO off (echoed input corrupts the protocol) and ICANON
    # off (canonical mode drops bytes past MAX_CANON with BEL spam; VMIN/VTIME make non-canonical reads
    # return per byte; ISIG stays on so ^C still interrupts)
    output_state = _tty_clear(sys.__stdout__, tty.OFLAG, termios.ONLCR)
    echo_state = _tty_clear(sys.stdin, tty.LFLAG, termios.ECHO | termios.ICANON, {termios.VMIN: 1, termios.VTIME: 0})
    delim = _new_delim()
    print(info, flush=True)
    print(f"<stream-protocol>\nOne-line request: send the line. Each response is an acknowledgement line '.' (request accepted, not complete), the rendered output, then the session delimiter line.\n"
        f"Multiline request (any multi-line cell, including %% cell magics), shown indented -- send it flush-left:\n"
        f"    --\n    <complete cell>\n    {delim}\n"
        f"No IPython prompt, no %cpaste, no invented terminators. A blank line is an empty request, so it doubles as an idle poll. Send 'exit' to end; a fresh process is the restart.\n</stream-protocol>", flush=True)
    print(_MARKER, flush=True)
    _write_response(delim)
    try:
        while True:
            line = _next_line(sys.stdin)
            if not line: break
            line = line.rstrip("\n")
            if line == delim:
                _write_response(delim, fmt_error("protocol-error", "no multiline request is open: start one with a bare `--` line"))
                continue
            if line == _MULTILINE:
                code, err = _read_block(sys.stdin, delim)
                if err:
                    _write_response(delim, fmt_error("protocol-error", err))
                    continue
            elif line.startswith('%%'):
                _write_response(delim, fmt_error("protocol-error",
                    f"a %% cell magic needs a multiline request; send it as (flush-left):\n    --\n    {line}\n    <rest of cell>\n    {delim}"))
                continue
            else: code = line
            print(".", flush=True)
            try: body = execute(code)
            except BaseException: body = fmt_error("internal-error", traceback.format_exc())
            _write_response(delim, body)
            if should_exit and should_exit(): break
    finally:
        _restore_termios(echo_state)
        _restore_termios(output_state)

`main` bridges the sync protocol loop to asyncio: the event loop runs in a background thread, each request crosses with `run_coroutine_threadsafe`, and a Ctrl-C while waiting becomes an `interrupt` on the kernel. The `connect` result — kernel id and startup banner — is the info block the protocol announces before the delimiter.


In [ ]:
#| export
_EXITS = ('exit', 'exit()', 'quit', 'quit()')

@call_parse
def main(
    host:str='',    # Gateway: empty for the local default, a `gateways.toml` name, or a URL
    kernel:str='',  # Kernel id (or unique prefix) to attach to; empty creates a kernel, stopped again on exit
):
    "The `clikernel` console script: the stream protocol over one gateway kernel"
    signal.signal(signal.SIGINT, signal.default_int_handler)
    print("please wait, loading...", flush=True)
    loop = asyncio.new_event_loop()
    threading.Thread(target=loop.run_forever, daemon=True).start()
    def run(coro): return asyncio.run_coroutine_threadsafe(coro, loop).result()
    c = Client()
    info = run(c.connect(host, kernel))
    stop = False
    def execute(code):
        nonlocal stop
        if code.strip() in _EXITS:
            stop = True
            return ''
        fut = asyncio.run_coroutine_threadsafe(c.execute(code), loop)
        while True:
            try: return fut.result()
            except KeyboardInterrupt: asyncio.run_coroutine_threadsafe(c.interrupt(), loop)
    try: serve_stream(execute, info=info, should_exit=lambda: stop)
    finally:
        if not kernel: run(c.stop())   # created for this run, cleaned up by this run
        run(c.aclose())
        loop.call_soon_threadsafe(loop.stop)


End to end over real pipes: the installed `clikernel` script against a live gateway. The handshake is the banner (the `connect` result), the recipe block, the marker line, then a delimiter; each request answers with the ack, the body, and the delimiter. On exit, the kernel this run created is stopped — but attaching with `--kernel` leaves the kernel running:


In [ ]:
g = start_gateway()
env = os.environ | {'CLIKERNEL_HOST': g.url, 'XDG_CONFIG_HOME': str(Path(tempfile.mkdtemp()))}

async def read_until(stream, delim):
    lines = []
    while True:
        line = (await asyncio.wait_for(stream.readline(), 30)).decode().rstrip('\n')
        if line == delim: return lines
        lines.append(line)

cmd = shutil.which('clikernel')
assert cmd, 'clikernel script not installed: run `uv sync`'
p = await asyncio.create_subprocess_exec(cmd, env=env,
    stdin=asyncio.subprocess.PIPE, stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.PIPE)
banner = []
while (line := (await asyncio.wait_for(p.stdout.readline(), 60)).decode().rstrip('\n')) != _MARKER: banner.append(line)
delim = (await p.stdout.readline()).decode().rstrip('\n')
assert any('created kernel' in l for l in banner)
delim[:2]


'--'

In [ ]:
async def ask(code):
    "One request/response cycle: send, read the ack, read to the delimiter"
    p.stdin.write((code+'\n').encode())
    await p.stdin.drain()
    ack = (await p.stdout.readline()).decode().rstrip('\n')
    test_eq(ack, '.')
    return '\n'.join(await read_until(p.stdout, delim))

test_eq(await ask('x = 21'), '')
test_eq(await ask('x*2'), '42')
p.stdin.write(f'--\n%%time\ny = x + 1\n{delim}\n'.encode())
await p.stdin.drain()
(await p.stdout.readline())
multi = '\n'.join(await read_until(p.stdout, delim))
assert 'CPU times' in multi
p.stdin.write(b'exit\n')
await p.stdin.drain()
await p.wait()
test_eq(httpx.get(f'{g.url}/api/kernels').json(), [])   # created-on-start means stopped-on-exit


And attach mode: the kernel exists before the CLI and survives it, state intact.

In [ ]:
kid = httpx.post(f'{g.url}/api/kernels').json()['id']
p = await asyncio.create_subprocess_exec(cmd, '--kernel', kid[:8], env=env,
    stdin=asyncio.subprocess.PIPE, stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.PIPE)
while (await asyncio.wait_for(p.stdout.readline(), 60)).decode().rstrip('\n') != _MARKER: pass
delim = (await p.stdout.readline()).decode().rstrip('\n')
test_eq(await ask('z = 99'), '')
p.stdin.write(b'exit\n')
await p.stdin.drain()
await p.wait()
test_eq([k['id'] for k in httpx.get(f'{g.url}/api/kernels').json()], [kid])  # attach never kills
httpx.delete(f'{g.url}/api/kernels/{kid}').status_code

204

In [ ]:
#|hide
g.stop()

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()